# Système de recommandation — Approche Collaborative Filtering (CF)

Principe : recommander des articles appréciés par des utilisateurs aux goûts similaires.

On utilise la librairie `implicit` qui est conçue pour les **feedbacks implicites** (clics, vues) — contrairement à `surprise` qui attend des notes explicites (1 à 5 étoiles).

Algorithme utilisé : **ALS (Alternating Least Squares)** — il factorise la matrice user-item en deux matrices de dimension réduite représentant les goûts des utilisateurs et les caractéristiques des articles.

Étapes :
1. Chargement des données
2. Construction de la matrice user-item
3. Entraînement du modèle ALS
4. Génération des recommandations (top 5)
5. Évaluation — Hit Rate@5

## 1. Chargement des données

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

DATA_DIR = "../news-portal-user-interactions-by-globocom"

clicks = pd.read_parquet(f"{DATA_DIR}/clicks_full.parquet")
print("Clics :", clicks.shape)
clicks.head()

Clics : (2988181, 12)


,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
1,0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
2,1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
3,1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
4,2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


## 2. Construction de la matrice user-item

Le CF repose sur une matrice où :
- chaque **ligne** = un utilisateur
- chaque **colonne** = un article
- chaque **valeur** = nombre de fois que l'utilisateur a cliqué sur cet article

Cette matrice est très **creuse** (sparse) : la plupart des cases valent 0 car chaque utilisateur n'a cliqué que sur une infime fraction des articles. On utilise donc un format sparse (`csr_matrix`) pour économiser la mémoire.

In [2]:
# Encodage des ids en indices entiers consécutifs
user_ids = clicks["user_id"].unique()
article_ids = clicks["click_article_id"].unique()

user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
article_to_index = {article_id: idx for idx, article_id in enumerate(article_ids)}
index_to_article = {idx: article_id for article_id, idx in article_to_index.items()}

print(f"Utilisateurs : {len(user_ids)}")
print(f"Articles uniques cliqués : {len(article_ids)}")

Utilisateurs : 322897
Articles uniques cliqués : 46033


In [3]:
def build_user_item_matrix(clicks_df: pd.DataFrame, user_to_index: dict, article_to_index: dict) -> csr_matrix:
    """Build a sparse user-item interaction matrix from click data.
    
    Args:
        clicks_df: DataFrame containing user-article click interactions.
        user_to_index: Mapping from user_id to row index.
        article_to_index: Mapping from article_id to column index.
    
    Returns:
        Sparse matrix of shape (n_users, n_articles) with click counts as values.
    """
    click_counts = clicks_df.groupby(["user_id", "click_article_id"]).size().reset_index(name="click_count")
    
    row_indices = click_counts["user_id"].map(user_to_index).values
    col_indices = click_counts["click_article_id"].map(article_to_index).values
    values = click_counts["click_count"].values
    
    return csr_matrix(
        (values, (row_indices, col_indices)),
        shape=(len(user_to_index), len(article_to_index))
    )

user_item_matrix = build_user_item_matrix(clicks, user_to_index, article_to_index)
print(f"Shape de la matrice user-item : {user_item_matrix.shape}")
print(f"Densité : {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")

Shape de la matrice user-item : (322897, 46033)
Densité : 0.000199


## 3. Entraînement du modèle ALS

ALS (Alternating Least Squares) est un algorithme de **factorisation de matrices**. Il décompose la matrice user-item en deux matrices plus petites :
- une matrice **utilisateurs** (user_factors) : représente les goûts de chaque user en N dimensions
- une matrice **articles** (item_factors) : représente les caractéristiques de chaque article en N dimensions

Le produit de ces deux matrices reconstitue approximativement la matrice originale.

In [4]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=50,
    iterations=20,
    regularization=0.1,
    random_state=42
)

# On passe user_item_matrix directement — les lignes sont les users
model.fit(user_item_matrix)

print("Modèle entraîné")
print(f"User factors shape : {model.user_factors.shape}")
print(f"Item factors shape : {model.item_factors.shape}")

  0%|          | 0/20 [00:00<?, ?it/s]

Modèle entraîné
User factors shape : (322897, 50)
Item factors shape : (46033, 50)


## 4. Génération des recommandations

In [9]:
def recommend_articles_cf(
    user_id: int,
    model: AlternatingLeastSquares,
    user_item_matrix: csr_matrix,
    user_to_index: dict,
    index_to_article: dict,
    top_n: int = 5
) -> list[int]:
    """Recommend articles using ALS collaborative filtering.
    
    Args:
        user_id: The user identifier.
        model: Trained ALS model.
        user_item_matrix: Sparse user-item matrix (n_users, n_articles).
        user_to_index: Mapping from user_id to row index.
        index_to_article: Mapping from column index to article_id.
        top_n: Number of recommendations to return.
    
    Returns:
        List of recommended article_ids.
    """
    if user_id not in user_to_index:
        return []
    
    user_index = user_to_index[user_id]
    item_indices, scores = model.recommend(
        user_index,
        user_item_matrix[user_index],
        N=top_n,
        filter_already_liked_items=True
    )
    return [int(index_to_article[int(idx)]) for idx in item_indices]

# Test sur un utilisateur exemple
sample_user_id = clicks["user_id"].iloc[0]
reco_cf = recommend_articles_cf(sample_user_id, model, user_item_matrix, user_to_index, index_to_article)
print(f"Top 5 recommandations CF pour l'utilisateur {sample_user_id} :")
print(reco_cf)

Top 5 recommandations CF pour l'utilisateur 0 :
[293114, 284985, 272218, 160940, 95716]


## 5. Évaluation — Top 5 recommandations / Hit Rate@5

* 20 itérations ALS

ALS est un algorithme itératif — à chaque itération il affine les matrices user_factors et item_factors en alternant entre les deux (d'où "Alternating"). 20 itérations est une valeur par défaut classique qui offre un bon compromis entre précision et temps de calcul. Au-delà de 20, les améliorations sont généralement marginales.

* 500 utilisateurs

On a 322 897 utilisateurs éligibles. 500 est un échantillon statistiquement représentatif pour estimer le hit rate@5 — l'erreur d'estimation reste faible. C'est un choix pragmatique pour l'évaluation offline.

In [10]:
def evaluate_cf(
    clicks_df: pd.DataFrame,
    user_to_index: dict,
    article_to_index: dict,
    index_to_article: dict,
    n_users: int = 500,
    top_n: int = 5
) -> float:
    """Evaluate CF recommendations using leave-one-out hit rate@N.
    
    Args:
        clicks_df: DataFrame containing user-article click interactions.
        user_to_index: Mapping from user_id to matrix row index.
        article_to_index: Mapping from article_id to matrix column index.
        index_to_article: Mapping from matrix column index to article_id.
        n_users: Number of users to sample for evaluation.
        top_n: Number of recommendations to generate.
    
    Returns:
        Hit Rate@N score between 0 and 1.
    """
    user_click_counts = clicks_df.groupby("user_id").size()
    eligible_users = user_click_counts[user_click_counts >= 3].index.tolist()
    sampled_users = pd.Series(eligible_users).sample(n=min(n_users, len(eligible_users)), random_state=42).tolist()
    
    hits = 0
    for user_id in sampled_users:
        # Tri par date de consultation : on masque le dernier article lu dans le temps
        user_clicks = (clicks_df[clicks_df["user_id"] == user_id]
                       .sort_values("click_timestamp")["click_article_id"].values)
        test_article = user_clicks[-1]  # leave-one-out : dernier article cliqué chronologiquement
        
        # Matrice d'entraînement sans le dernier clic
        train_clicks = clicks_df[
            ~((clicks_df["user_id"] == user_id) & (clicks_df["click_article_id"] == test_article))
        ]
        train_matrix = build_user_item_matrix(train_clicks, user_to_index, article_to_index)
        
        train_model = AlternatingLeastSquares(factors=50, iterations=20, regularization=0.1, random_state=42)
        train_model.fit(train_matrix)
        
        # générer le top_5 des reco pour ce user
        reco = recommend_articles_cf(user_id, train_model, train_matrix, user_to_index, index_to_article, top_n)
        # Hit binaire par utilisateur, 1 si l'article masqué est dans les reco, 0 sinon
        if test_article in reco:
            hits += 1
    
    hit_rate = hits / len(sampled_users)  # fraction d'utilisateurs pour lesquels la bonne réponse figure dans la liste des users de longueur K
    print(f"Hit Rate@{top_n} sur {len(sampled_users)} utilisateurs : {hit_rate:.4f}")
    return hit_rate

hit_rate_cf = evaluate_cf(clicks, user_to_index, article_to_index, index_to_article)

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Hit Rate@5 sur 500 utilisateurs : 0.1060


In [11]:
print(f"Hit Rate@5 CF : {hit_rate_cf:.4f}")

Hit Rate@5 CF : 0.1060


* scores de Hit Rate@5 :
    - Content-Based	: 0.0040
    - Collaborative Filtering : 0.1060

* Le CF est 26 fois plus précis que le content-based sur cette métrique. C'est un résultat cohérent car le CF exploite les comportements collectifs de 322 897 utilisateurs, ce qui lui donne beaucoup plus de signal pour prédire le prochain clic.
Mais ce score ne signifie pas que le CF est toujours meilleur. 

* le contentbased reste indispendsable malgré son score de précision@5 faible en raison du cold-start article, et aussi du cold-start utilisateur : pour les nouveaux users sans historique, le CF ne peut rien faire alors que le content-based démarre dès le premier clic.

* En pratique, les systèmes de recommandation en production combinent souvent les deux approches — c'est ce qu'on appelle un système hybride :
    - Si l'utilisateur a suffisamment de clics => CF
    - sinon => content-based

à avoir en tête au sujet de l'évolution du système.